# O-HAZE inference cost profile on Kaggle

Profile the three fixed Stage 4 checkpoints (seeds 17, 43, 73) at 256×256 and 512×512, batch size 1, using radiometric O-HAZE inputs. Each checkpoint/resolution/mode gets 20 warm-up and 100 timed forward passes, cycling through three deterministically selected O-HAZE hazy images.

Two scopes are timed: **model-only** uses preloaded radiometric tensors held on CPU, so GPU runs include TensorFlow's host-to-device placement/copy when required, the full-model forward pass, and synchronized output readback; **end-to-end** includes image file read, resize, RGB conversion, radiometric scaling, and model forward pass. Both synchronize model output before stopping the timer. Model loading, output writing, and metric calculation are excluded. TensorFlow allocator peak is recorded after warm-up, during the timed phase, when a GPU is available.

This is a hardware- and runtime-specific engineering profile, not a deployment-readiness test. It does not measure cold storage behavior, memory used outside TensorFlow's GPU allocator, concurrency, or end-user latency. No training occurs. Run in Kaggle with the package and O-HAZE dataset mounted.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import time

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

PACKAGE = Path("/kaggle/input/datasets/randomguy28/reviewer-test-packge/reviewer_test_package")
OHAZE_ROOT = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY")
HAZY_DIR = OHAZE_ROOT / "hazy"
GT_DIR = OHAZE_ROOT / "GT"
OUTPUT_ROOT = Path("/kaggle/working/ohaze_inference_cost_profile")
SCRIPT_ROOT = PACKAGE / "code"
CHECKPOINTS = {
    seed: PACKAGE / f"models/full/seed{seed}/stage4_best.keras"
    for seed in (17, 43, 73)
}
RESOLUTIONS = (256, 512)
BATCH_SIZE = 1
WARMUP_RUNS = 20
TIMED_RUNS = 100
N_SAMPLE_IMAGES = 3

assert PACKAGE.is_dir(), f"Check Kaggle package path: {PACKAGE}"
assert HAZY_DIR.is_dir(), f"Check O-HAZE hazy directory: {HAZY_DIR}"
assert GT_DIR.is_dir(), f"Check O-HAZE GT directory: {GT_DIR}"
assert all(path.is_file() for path in CHECKPOINTS.values()), "Missing Stage 4 checkpoint."

sys.path.insert(0, str(SCRIPT_ROOT))
from eca_ldnet.data import get_gt_path, preprocess_hazy
from eca_ldnet.layers import CUSTOM_OBJECTS

def sha256_file(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

image_exts = {".png", ".jpg", ".jpeg", ".bmp"}
pairs, unmatched = [], []
for hazy_path in sorted(p for p in HAZY_DIR.iterdir() if p.suffix.lower() in image_exts):
    gt_path = get_gt_path(hazy_path.name, str(GT_DIR))
    if gt_path is None:
        unmatched.append(hazy_path.name)
    else:
        pairs.append({"image": hazy_path.name, "hazy_path": str(hazy_path), "clean_path": str(gt_path)})
assert len(pairs) == 45 and not unmatched, f"Expected 45 matched O-HAZE pairs; got {len(pairs)}, unmatched={unmatched[:5]}"
# Deterministic sample: first, middle, and last image in sorted filename order.
sample_indices = sorted(set((0, len(pairs) // 2, len(pairs) - 1)))
sample_pairs = [pairs[i] for i in sample_indices]
assert len(sample_pairs) == N_SAMPLE_IMAGES
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Profile samples:", [p["image"] for p in sample_pairs])
print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))

## Record runtime hardware and software

TensorFlow uses its normal device placement. If Kaggle has a GPU enabled and TensorFlow sees it, model calls use TensorFlow's configured accelerator. Keep this runtime configuration unchanged during the benchmark.

In [ ]:
gpu_devices = tf.config.list_physical_devices("GPU")
gpu_details = []
for device in gpu_devices:
    try:
        gpu_details.append({"name": device.name, **tf.config.experimental.get_device_details(device)})
    except Exception as exc:
        gpu_details.append({"name": device.name, "details_error": str(exc)})
try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True, timeout=10,
    ).stdout.strip().splitlines()
except Exception:
    nvidia_smi = []

runtime_metadata = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "tensorflow": tf.__version__,
    "tensorflow_build_info": tf.sysconfig.get_build_info(),
    "logical_cpu_count": os.cpu_count(),
    "tensorflow_gpu_devices": gpu_details,
    "nvidia_smi": nvidia_smi,
    "default_device_placement": "TensorFlow default placement; GPU if available and configured",
}
print(json.dumps(runtime_metadata, indent=2, default=str))

## Run fixed-checkpoint timing profile

Each configuration gets 20 warm-ups and 100 timed calls, cycling across the three samples. Synchronous `.numpy()` forces completion before the timer stops. End-to-end timing reopens the image file on every pass; filesystem cache state is not controlled.

In [ ]:
def read_radiometric(path, size):
    """Match package ordering: read BGR, resize, RGB, float [0,1], radiometric clip."""
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(path)
    image = cv2.resize(image, (size, size))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return preprocess_hazy(image, protocol="radiometric")

def synchronize_call(model, batch):
    output = model(batch, training=False)
    # Force device completion; avoids measuring only asynchronous GPU launch time.
    _ = output.numpy()

def reset_gpu_peak():
    if not gpu_devices:
        return False
    try:
        tf.config.experimental.reset_memory_stats("GPU:0")
        return True
    except Exception:
        return False

def read_gpu_peak_bytes():
    if not gpu_devices:
        return None
    try:
        return int(tf.config.experimental.get_memory_info("GPU:0")["peak"])
    except Exception:
        return None

raw_timing_rows = []
summary_rows = []
checkpoint_hashes = {seed: sha256_file(path) for seed, path in CHECKPOINTS.items()}

for seed, checkpoint in CHECKPOINTS.items():
    print(f"Loading checkpoint seed {seed}: {checkpoint.name}")
    model = tf.keras.models.load_model(str(checkpoint), custom_objects=CUSTOM_OBJECTS, compile=False)
    for size in RESOLUTIONS:
        arrays = [read_radiometric(row["hazy_path"], size) for row in sample_pairs]
        # Keep preloaded inputs on host RAM. On GPU runs, the model call then
        # includes TensorFlow's host-to-device placement/copy when required.
        with tf.device("/CPU:0"):
            batches = [tf.convert_to_tensor(image[None, ...], dtype=tf.float32) for image in arrays]

        for mode in ("model_only", "end_to_end"):
            print(f"  seed={seed}, resolution={size}, mode={mode}")
            def run_once(iteration):
                sample_index = iteration % len(sample_pairs)
                if mode == "model_only":
                    batch = batches[sample_index]
                else:
                    image = read_radiometric(sample_pairs[sample_index]["hazy_path"], size)
                    with tf.device("/CPU:0"):
                        batch = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)
                synchronize_call(model, batch)

            for warm_index in range(WARMUP_RUNS):
                run_once(warm_index)
            memory_reset_ok = reset_gpu_peak()

            elapsed_ms = []
            sample_names = []
            for run_index in range(TIMED_RUNS):
                sample_index = run_index % len(sample_pairs)
                start = time.perf_counter()
                run_once(run_index + WARMUP_RUNS)
                elapsed_ms.append((time.perf_counter() - start) * 1000.0)
                sample_names.append(sample_pairs[(run_index + WARMUP_RUNS) % len(sample_pairs)]["image"])
            peak_bytes = read_gpu_peak_bytes() if memory_reset_ok else None

            for run_index, (latency, sample_name) in enumerate(zip(elapsed_ms, sample_names), start=1):
                raw_timing_rows.append({
                    "seed": seed, "checkpoint": str(checkpoint), "resolution": size,
                    "batch_size": BATCH_SIZE, "mode": mode, "run": run_index,
                    "sample_image": sample_name, "latency_ms": latency,
                })
            summary_rows.append({
                "seed": seed, "checkpoint": str(checkpoint), "resolution": size,
                "batch_size": BATCH_SIZE, "mode": mode, "warmup_runs": WARMUP_RUNS,
                "timed_runs": TIMED_RUNS,
                "median_ms": float(np.median(elapsed_ms)),
                "p95_ms": float(np.percentile(elapsed_ms, 95)),
                "mean_ms": float(np.mean(elapsed_ms)),
                "images_per_second_from_mean": float(1000.0 / np.mean(elapsed_ms)),
                "tf_gpu_allocator_peak_bytes": peak_bytes,
                "memory_reset_supported": memory_reset_ok,
            })
        del batches, arrays
    del model
    tf.keras.backend.clear_session()

raw_timings = pd.DataFrame(raw_timing_rows)
summary = pd.DataFrame(summary_rows)
assert len(raw_timings) == len(CHECKPOINTS) * len(RESOLUTIONS) * 2 * TIMED_RUNS
assert summary.groupby(["seed", "resolution", "mode"])["timed_runs"].first().eq(TIMED_RUNS).all()
raw_path = OUTPUT_ROOT / "raw_timings.csv"
summary_path = OUTPUT_ROOT / "summary_by_checkpoint.csv"
raw_timings.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)
display(summary.sort_values(["resolution", "mode", "seed"]).round(3))

## Aggregate view and saved outputs

Summary rows per seed remain the primary report. The aggregate plot pools the 100 timing observations across three checkpoints for each resolution and timing scope; this describes this checkpoint set on this Kaggle runtime. GPU memory panel reports maximum TensorFlow allocator peak among the three checkpoints for each scope and resolution. A blank/annotated panel means no GPU allocator reading was available.

In [ ]:
pooled_rows = []
for (size, mode), group in raw_timings.groupby(["resolution", "mode"]):
    values = group["latency_ms"].to_numpy()
    memory_values = summary.loc[
        (summary["resolution"] == size) & (summary["mode"] == mode),
        "tf_gpu_allocator_peak_bytes",
    ].dropna()
    pooled_rows.append({
        "resolution": int(size), "mode": mode, "n_timed_calls": int(len(values)),
        "median_ms": float(np.median(values)), "p95_ms": float(np.percentile(values, 95)),
        "mean_ms": float(np.mean(values)),
        "images_per_second_from_mean": float(1000.0 / np.mean(values)),
        "max_tf_gpu_allocator_peak_bytes_across_checkpoints": int(memory_values.max()) if len(memory_values) else None,
    })
pooled_summary = pd.DataFrame(pooled_rows)
pooled_path = OUTPUT_ROOT / "summary_pooled_checkpoint_calls.csv"
pooled_summary.to_csv(pooled_path, index=False)
display(pooled_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
latency_labels, latency_medians, latency_upper, latency_colors = [], [], [], []
for size in RESOLUTIONS:
    for mode in ("model_only", "end_to_end"):
        row = pooled_summary.loc[
            (pooled_summary["resolution"] == size) & (pooled_summary["mode"] == mode)
        ].iloc[0]
        latency_labels.append(f"{size}\n{mode.replace('_', ' ')}")
        latency_medians.append(row["median_ms"])
        latency_upper.append(max(row["p95_ms"] - row["median_ms"], 0))
        latency_colors.append({"model_only":"#4472C4", "end_to_end":"#ED7D31"}[mode])
x_positions = np.arange(len(latency_labels))
axes[0].bar(
    x_positions, latency_medians,
    yerr=np.vstack([np.zeros(len(latency_upper)), latency_upper]),
    capsize=3, color=latency_colors,
)
axes[0].set_xticks(x_positions, latency_labels)
axes[0].set(ylabel="Latency (ms)", title="Batch-1 median; whisker extends to p95")
axes[0].grid(axis="y", alpha=0.25)
axes[0].tick_params(axis="x", labelsize=8)

if pooled_summary["max_tf_gpu_allocator_peak_bytes_across_checkpoints"].notna().any():
    for mode in ("model_only", "end_to_end"):
        part = pooled_summary.loc[pooled_summary["mode"] == mode].sort_values("resolution")
        labels = [f"{int(r)}\n{mode.replace('_',' ')}" for r in part["resolution"]]
        values = part["max_tf_gpu_allocator_peak_bytes_across_checkpoints"].fillna(0).to_numpy() / (1024**3)
        axes[1].bar(labels, values, color={"model_only":"#4472C4", "end_to_end":"#ED7D31"}[mode], label=mode.replace("_", " "))
    axes[1].set(ylabel="Peak TensorFlow GPU allocator (GiB)", title="Maximum observed across checkpoints")
    axes[1].grid(axis="y", alpha=0.25)
    axes[1].tick_params(axis="x", labelsize=8)
else:
    axes[1].text(0.5, 0.5, "No GPU allocator data available", ha="center", va="center")
    axes[1].set_axis_off()
fig.tight_layout()
plot_path = OUTPUT_ROOT / "inference_cost_profile.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()

manifest = {
    "dataset": "O-HAZE",
    "matched_pair_count": len(pairs),
    "profile_sample_images": [row["image"] for row in sample_pairs],
    "checkpoints": {str(seed): str(path) for seed, path in CHECKPOINTS.items()},
    "checkpoint_sha256": {str(seed): digest for seed, digest in checkpoint_hashes.items()},
    "resolutions": list(RESOLUTIONS), "batch_size": BATCH_SIZE,
    "warmup_runs_per_configuration": WARMUP_RUNS,
    "timed_runs_per_configuration": TIMED_RUNS,
    "timing_modes": {
        "model_only": "Preloaded radiometric tensor held on CPU; includes model invocation, any TensorFlow host-to-device placement/copy, full-model forward pass, and synchronized output readback.",
        "end_to_end": "OpenCV file read, resize, BGR-to-RGB, float scaling/radiometric preprocessing, TensorFlow full-model forward pass; output synchronized.",
    },
    "excluded": ["checkpoint loading", "output serialization", "metric calculation", "cold-storage guarantees", "concurrent requests"],
    "memory_scope": "TensorFlow GPU allocator peak during timed phase after warm-up; excludes host RAM and other GPU allocations; null if unavailable.",
    "filesystem_cache": "Not controlled; end-to-end repeated reads may be served from OS cache.",
    "training_performed": False,
    "interpretation": "Hardware/runtime-specific engineering characterization; not a deployment-readiness claim.",
    "runtime": runtime_metadata,
    "outputs": {
        "raw_timings_csv": str(raw_path),
        "summary_by_checkpoint_csv": str(summary_path),
        "pooled_summary_csv": str(pooled_path),
        "plot_png": str(plot_path),
    },
}
manifest_path = OUTPUT_ROOT / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
print("Saved raw timings:", raw_path)
print("Saved summaries:", summary_path, pooled_path)
print("Saved plot and manifest:", plot_path, manifest_path)

## Package outputs

Create a ZIP of all files in `/kaggle/working/ohaze_inference_cost_profile/`. The archive is saved beside that folder.

In [ ]:
import shutil

archive_path = shutil.make_archive(
    str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
assert Path(archive_path).is_file(), f"ZIP was not created: {archive_path}"
print(f"Created ZIP: {archive_path}")